<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>Generative AI for Business Applications</center></font>
<center><font size=6>Agent Communication Protocol</center></font>

# 1. What is ACP (Agent Communication Protocol)?

## The Problem

In the previous notebook we saw how **MCP (Model Context Protocol)** lets an LLM call external tools — weather APIs, country databases, holiday calendars. MCP is great for *vertical* integration: one LLM reaches down into tools.

But what happens when you have **multiple AI agents** that need to collaborate? If an Advisor agent wants to ask a Weather agent for data, MCP doesn't help — MCP connects an LLM to tools, not an agent to another agent.

## The Solution: ACP

**Agent Communication Protocol (ACP)** is an open standard for **agent-to-agent communication**. Think of it as a **phone network for AI agents** — any agent can discover, call, and get responses from any other agent over standard HTTP.

### Architecture

```
┌──────────────┐     ACP/HTTP      ┌──────────────┐     ACP/HTTP     ┌──────────────┐
│   Notebook   │ ───────────────►  │   Advisor    │ ────────────────► │   Data       │ ──► Real APIs
│   (Client)   │   port 8001       │    Agent     │   port 8000      │   Agents     │
└──────────────┘                   └──────────────┘                   └──────────────┘
```

### MCP vs ACP — When to Use Which

| | **MCP** | **ACP** |
|---|---------|--------|
| **Direction** | Vertical: LLM → Tools | Horizontal: Agent → Agent |
| **Unit of work** | A function call (get_weather) | A full agent run (plan → research → answer) |
| **Who decides?** | The LLM picks which tool to call | The receiving agent decides how to fulfill the request |
| **Discovery** | Client reads tool schemas | Client lists agent manifests via `GET /agents` |
| **Transport** | stdio / SSE | Standard HTTP |
| **Best for** | Giving one LLM access to data & actions | Composing autonomous agents into workflows |

In this notebook, we'll build the **same travel research** use case from the MCP notebook — but this time the intelligence is distributed across **collaborating agents** instead of one LLM with tools.

# 2. Setup

## Install Dependencies

In [12]:
!pip install -U langchain langchain-community langchain-openai acp-sdk==1.0.3 httpx "uvicorn<0.36"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: openai
    Found existing installation: openai 1.109.1
    Uninstalling openai-1.109.1:
      Successfully

**Note:**
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells starting from the next cell onward.

## Imports

In [1]:
import json
import os
import subprocess
import sys
import time
import asyncio

from pathlib import Path
from getpass import getpass


def load_openai_config(required_for: str) -> dict:
    config_path = Path("config.json")
    config = {}
    if config_path.exists():
        config = json.loads(config_path.read_text())

    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    def read_value(key: str, default: str | None = None) -> str | None:
        if config.get(key):
            return config[key]
        env_value = os.environ.get(key)
        if env_value:
            return env_value
        if userdata is not None:
            try:
                secret_value = userdata.get(key)
            except Exception:
                secret_value = None
            if secret_value:
                return secret_value
        return default

    openai_api_key = read_value("OPENAI_API_KEY")
    openai_api_base = read_value("OPENAI_API_BASE", "https://api.openai.com/v1")

    if not openai_api_key:
        openai_api_key = getpass("Enter your OPENAI_API_KEY: ").strip()

    if not openai_api_key:
        raise ValueError(f"OPENAI_API_KEY is required to run {required_for}.")

    config = {
        "OPENAI_API_KEY": openai_api_key,
        "OPENAI_API_BASE": openai_api_base,
    }
    config_path.write_text(json.dumps(config, indent=2))

    print(f"Saved OpenAI config to {config_path.resolve()}")
    print(f"Using OPENAI_API_BASE={openai_api_base}")
    return config

config = load_openai_config("the ACP advisor agent")


Saved OpenAI config to /content/config.json
Using OPENAI_API_BASE=https://aibe.mygreatlearning.com/openai/v1


In [2]:
import nest_asyncio
nest_asyncio.apply()

# 3. Building the Data Agents

Our first ACP server hosts **three specialized data agents**, each wrapping one free public API:

| Agent | API | What it does |
|-------|-----|-------------|
| `weather_agent` | Open-Meteo | Geocodes a city, returns current weather + 7-day forecast |
| `country_agent` | REST Countries | Returns country profile (capital, languages, currency) |
| `holiday_agent` | Nager.Date | Lists public holidays for a country code + year |

Unlike MCP tools, these agents are **autonomous** — they receive a plain-text request and decide internally how to fulfill it. No LLM is needed on this server.

In [15]:
%%writefile data_agent.py
# ============================================================
# ACP Data Agent Server
# Three specialized agents wrapping free public APIs
# ============================================================

import asyncio
from collections.abc import AsyncGenerator

import httpx
from acp_sdk.models import Message, MessagePart
from acp_sdk.server import Context, RunYield, RunYieldResume, Server

server = Server()


# ── Private helpers ───────────────────────────────────────────

async def _geocode(city: str) -> dict | None:
    """Look up latitude/longitude for a city name."""
    async with httpx.AsyncClient() as client:
        resp = await client.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1}
        )
        data = resp.json()
        if "results" not in data:
            return None
        r = data["results"][0]
        return {
            "name": r["name"],
            "country": r.get("country", "Unknown"),
            "lat": r["latitude"],
            "lon": r["longitude"]
        }


def _extract_text(input: list[Message]) -> str:
    """Extract plain text from ACP input messages."""
    parts = []
    for msg in input:
        for part in msg.parts:
            if hasattr(part, "content") and isinstance(part.content, str):
                parts.append(part.content)
    return " ".join(parts).strip()


# ── Agent 1: Weather ──────────────────────────────────────────

@server.agent(
    name="weather_agent",
    description="Get current weather and 7-day forecast for a city. Send the city name as plain text."
)
async def weather_agent(
    input: list[Message], context: Context
) -> AsyncGenerator[RunYield, RunYieldResume]:
    city = _extract_text(input)
    if not city:
        yield MessagePart(content="Please provide a city name.", content_type="text/plain")
        return

    geo = await _geocode(city)
    if not geo:
        yield MessagePart(content=f"Could not find city: {city}", content_type="text/plain")
        return

    async with httpx.AsyncClient() as client:
        resp = await client.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": geo["lat"],
                "longitude": geo["lon"],
                "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",
                "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
                "timezone": "auto",
                "forecast_days": 7
            }
        )
        data = resp.json()

    current = data["current"]
    daily = data["daily"]

    report = f"Weather Report: {geo['name']}, {geo['country']}\n"
    report += "=" * 50 + "\n\n"
    report += f"Current Conditions:\n"
    report += f"  Temperature: {current['temperature_2m']}\u00b0C\n"
    report += f"  Humidity:    {current['relative_humidity_2m']}%\n"
    report += f"  Wind Speed:  {current['wind_speed_10m']} km/h\n\n"
    report += f"7-Day Forecast:\n"
    for i in range(7):
        report += (
            f"  {daily['time'][i]}: "
            f"{daily['temperature_2m_min'][i]}\u00b0C \u2013 {daily['temperature_2m_max'][i]}\u00b0C, "
            f"rain {daily['precipitation_sum'][i]} mm\n"
        )

    yield MessagePart(content=report, content_type="text/plain")


# ── Agent 2: Country Info ─────────────────────────────────────

@server.agent(
    name="country_agent",
    description="Get detailed country information (capital, population, languages, currency). Send the country name as plain text."
)
async def country_agent(
    input: list[Message], context: Context
) -> AsyncGenerator[RunYield, RunYieldResume]:
    country_name = _extract_text(input)
    if not country_name:
        yield MessagePart(content="Please provide a country name.", content_type="text/plain")
        return

    async with httpx.AsyncClient() as client:
        resp = await client.get(
            f"https://restcountries.com/v3.1/name/{country_name}",
            params={"fullText": "false"}
        )
        if resp.status_code != 200:
            yield MessagePart(content=f"Country not found: {country_name}", content_type="text/plain")
            return
        countries = resp.json()

    c = countries[0]
    languages = ", ".join(c.get("languages", {}).values()) or "N/A"
    currencies = ", ".join(
        f"{v['name']} ({v.get('symbol', '')})" for v in c.get("currencies", {}).values()
    ) or "N/A"
    capitals = ", ".join(c.get("capital", [])) or "N/A"

    report = f"Country Profile: {c['name']['common']}\n"
    report += "=" * 50 + "\n\n"
    report += f"  Official Name: {c['name'].get('official', 'N/A')}\n"
    report += f"  Capital:       {capitals}\n"
    report += f"  Region:        {c.get('region', 'N/A')} / {c.get('subregion', 'N/A')}\n"
    report += f"  Population:    {c.get('population', 0):,}\n"
    report += f"  Languages:     {languages}\n"
    report += f"  Currencies:    {currencies}\n"
    report += f"  Timezone(s):   {', '.join(c.get('timezones', []))}\n"
    report += f"  Drives on:     {c.get('car', {}).get('side', 'N/A')} side\n"

    yield MessagePart(content=report, content_type="text/plain")


# ── Agent 3: Public Holidays ──────────────────────────────────

@server.agent(
    name="holiday_agent",
    description="Get public holidays for a country. Send a 2-letter ISO country code and year separated by a space (e.g. 'JP 2026')."
)
async def holiday_agent(
    input: list[Message], context: Context
) -> AsyncGenerator[RunYield, RunYieldResume]:
    text = _extract_text(input)
    parts = text.split()
    if len(parts) < 2:
        yield MessagePart(
            content="Please provide a country code and year, e.g. 'JP 2026'.",
            content_type="text/plain"
        )
        return

    country_code = parts[0].upper()
    year = parts[1]

    async with httpx.AsyncClient() as client:
        resp = await client.get(
            f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country_code}"
        )
        if resp.status_code != 200:
            yield MessagePart(
                content=f"Could not fetch holidays for {country_code} in {year}.",
                content_type="text/plain"
            )
            return
        holidays = resp.json()

    report = f"Public Holidays: {country_code} ({year})\n"
    report += "=" * 50 + "\n\n"
    for h in holidays:
        report += f"  {h['date']}  {h['localName']} ({h['name']})\n"
    report += f"\nTotal: {len(holidays)} public holidays\n"

    yield MessagePart(content=report, content_type="text/plain")


# ── Entry point ──────────────────────────────────────────────
if __name__ == "__main__":
    server.run(port=8000)

Overwriting data_agent.py


# 4. Building the Advisor Agent

The Advisor is an **LLM-powered agent** that acts as an intelligent orchestrator. When it receives a user query, it:

1. **Discovers** available data agents via ACP (`GET /agents`)
2. **Plans** which agents to call using the LLM (outputs a JSON plan)
3. **Executes** the plan by calling data agents via ACP Client
4. **Synthesizes** all collected data into a final advisory using the LLM

The Advisor never calls any API directly — it **delegates** to peer agents via ACP.

In [16]:
%%writefile advisor_agent.py
# ============================================================
# ACP Advisor Agent Server
# LLM-powered orchestrator that delegates to data agents
# ============================================================

import asyncio
import json
from collections.abc import AsyncGenerator

import httpx
from acp_sdk.client import Client
from acp_sdk.models import Message, MessagePart, Run
from acp_sdk.server import Context, RunYield, RunYieldResume, Server
from langchain_openai import ChatOpenAI

# ── LLM Configuration ─────────────────────────────────────────

with open("config.json") as f:
    config = json.load(f)

llm = ChatOpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config.get("OPENAI_API_BASE") or "https://api.openai.com/v1",
    model="gpt-4o-mini",
    temperature=0
)

DATA_AGENT_URL = "http://localhost:8000"

server = Server()


# ── Helpers ────────────────────────────────────────────────────

def _extract_text(input: list[Message]) -> str:
    """Extract plain text from ACP input messages."""
    parts = []
    for msg in input:
        for part in msg.parts:
            if hasattr(part, "content") and isinstance(part.content, str):
                parts.append(part.content)
    return " ".join(parts).strip()


async def discover_agents() -> list[dict]:
    """Discover available agents on the data agent server."""
    async with Client(base_url=DATA_AGENT_URL) as client:
        agents = []
        async for agent in client.agents():
            agents.append({
                "name": agent.name,
                "description": agent.description
            })
    return agents


async def run_agent_sync(base_url: str, agent_name: str, query: str) -> Run:
    """Call an ACP agent synchronously using explicit JSON body semantics."""
    message = Message(parts=[MessagePart(content=query, content_type="text/plain")])
    payload = {
        "agent_name": agent_name,
        "input": [message.model_dump(mode="json")],
        "mode": "sync",
    }
    async with httpx.AsyncClient(base_url=base_url, timeout=60) as client:
        response = await client.post("/runs", json=payload)
        response.raise_for_status()
    return Run.model_validate(response.json())


async def query_data_agent(agent_name: str, query: str) -> str:
    """Send a query to a data agent and return its text response."""
    run = await run_agent_sync(DATA_AGENT_URL, agent_name, query)
    texts = []
    for msg in run.output:
        for part in msg.parts:
            if hasattr(part, "content") and isinstance(part.content, str):
                texts.append(part.content)
    return "\n".join(texts)


# ── Advisor Agent ─────────────────────────────────────────────

@server.agent(
    name="advisor",
    description="Travel advisor that answers questions by consulting specialized data agents. Send any travel-related question as plain text."
)
async def advisor(
    input: list[Message], context: Context
) -> AsyncGenerator[RunYield, RunYieldResume]:
    user_query = _extract_text(input)

    # Step 1: Discover available data agents
    agents = await discover_agents()
    agent_descriptions = "\n".join(
        f"- {a['name']}: {a['description']}" for a in agents
    )

    # Step 2: Ask the LLM to plan which agents to call
    planning_prompt = f"""You are a travel advisor. A user has asked:

"{user_query}"

You have access to these data agents:
{agent_descriptions}

Decide which agents to call and what query to send each one.
Respond with ONLY a JSON object in this exact format:
{{
  "calls": [
    {{"agent": "agent_name", "query": "the query to send"}}
  ]
}}

Rules:
- Only use agents from the list above.
- For holiday_agent, send the query as "CC YYYY" (e.g. "JP 2026").
- You may call the same agent multiple times with different queries if needed.
- Include all agents needed to fully answer the user's question."""

    plan_response = await llm.ainvoke(planning_prompt)
    plan_text = plan_response.content

    # Parse the JSON plan
    try:
        # Handle markdown code fences if present
        if "```" in plan_text:
            plan_text = plan_text.split("```")[1]
            if plan_text.startswith("json"):
                plan_text = plan_text[4:]
        plan = json.loads(plan_text.strip())
    except (json.JSONDecodeError, IndexError):
        yield MessagePart(
            content=f"Failed to parse plan. Raw LLM output:\n{plan_response.content}",
            content_type="text/plain"
        )
        return

    # Step 3: Execute the plan — call data agents in parallel
    calls = plan.get("calls", [])
    if not calls:
        yield MessagePart(
            content="No agents needed for this query.",
            content_type="text/plain"
        )
        return

    tasks = [
        query_data_agent(call["agent"], call["query"])
        for call in calls
    ]
    results = await asyncio.gather(*tasks, return_exceptions=True)

    # Collect results
    collected = ""
    for call, result in zip(calls, results):
        collected += f"\n--- {call['agent']} (query: {call['query']}) ---\n"
        if isinstance(result, Exception):
            collected += f"Error: {result}\n"
        else:
            collected += f"{result}\n"

    # Step 4: Ask the LLM to synthesize a final answer
    synthesis_prompt = f"""You are a travel advisor. The user asked:

"{user_query}"

You consulted the following data agents and received these results:
{collected}

Write a helpful, well-structured travel advisory that answers the user's question.
Be concise but informative, and clearly synthesize the information instead of dumping raw results."""

    final_response = await llm.ainvoke(synthesis_prompt)
    yield MessagePart(content=final_response.content, content_type="text/plain")


# ── Entry point ──────────────────────────────────────────────
if __name__ == "__main__":
    server.run(port=8001)


Overwriting advisor_agent.py


# 5. Understanding the Architecture

Let's review what we just built:

### Components

| Component | File | Port | LLM? | Purpose |
|-----------|------|------|------|---------|
| Data Agents | `data_agent.py` | 8000 | No | Wrap free APIs as autonomous agents |
| Advisor Agent | `advisor_agent.py` | 8001 | Yes | Plan, delegate, and synthesize |
| Client | This notebook | — | No | Send queries, display results |

### Communication Flow

```
1. Client sends "Tell me about Japan" to Advisor (port 8001)
2. Advisor discovers data agents via GET /agents on port 8000
3. Advisor's LLM plans: call country_agent("Japan"), weather_agent("Tokyo"), holiday_agent("JP 2026")
4. Advisor calls all three data agents in parallel via ACP Client
5. Each data agent fetches from its API and returns structured text
6. Advisor's LLM synthesizes all results into a travel advisory
7. Client receives the final response
```

### How This Differs from MCP

In the MCP notebook, the **LLM directly called tools** like `get_weather()` and `get_country_info()`. Here:

- The **data agents are autonomous** — they decide internally how to fulfill requests
- The **Advisor delegates** instead of calling APIs — it sends natural language to peer agents
- **Agent discovery** replaces tool schemas — the Advisor asks "who's available?" at runtime
- **Each agent is independently deployable** — they could run on different machines

# 6. Starting the Agents

In [17]:
# Kill any stale processes on ports 8000 and 8001
!kill $(lsof -t -i:8000) 2>/dev/null; true
!kill $(lsof -t -i:8001) 2>/dev/null; true

def ensure_running(proc, name):
    if proc.poll() is not None:
        stdout, stderr = proc.communicate(timeout=1)
        raise RuntimeError(
            f"{name} exited early.\n\nSTDOUT:\n{stdout}\n\nSTDERR:\n{stderr}"
        )

# Launch both agent servers as background subprocesses
data_proc = subprocess.Popen(
    [sys.executable, "data_agent.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)
print(f"Data agent server started (PID: {data_proc.pid})")

# Give the data agent a moment to start before launching the advisor
time.sleep(3)
ensure_running(data_proc, "Data agent server")

advisor_proc = subprocess.Popen(
    [sys.executable, "advisor_agent.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)
print(f"Advisor agent server started (PID: {advisor_proc.pid})")

# Wait for both servers to be ready
time.sleep(5)
ensure_running(data_proc, "Data agent server")
ensure_running(advisor_proc, "Advisor agent server")
print("Both servers are ready.")

Data agent server started (PID: 2825)
Advisor agent server started (PID: 2838)
Both servers are ready.


In [18]:
# Final verification of agent discovery
import httpx
import asyncio

async def verify_servers():
    async with httpx.AsyncClient() as client:
        print("Data Agent Server (port 8000):")
        resp_data = await client.get("http://localhost:8000/agents")
        for agent in resp_data.json().get("agents", []):
            print(f"  - {agent['name']}")

        print("\nAdvisor Agent Server (port 8001):")
        resp_adv = await client.get("http://localhost:8001/agents")
        for agent in resp_adv.json().get("agents", []):
            print(f"  - {agent['name']}")

try:
    asyncio.run(verify_servers())
except Exception as e:
    print(f"Verification failed: {e}")

Data Agent Server (port 8000):
  - weather_agent
  - country_agent
  - holiday_agent

Advisor Agent Server (port 8001):
Verification failed: All connection attempts failed


# 7. Client Helper

The notebook client is trivially simple — it just sends a message to the Advisor agent and prints the response. All the intelligence lives in the agents, not the client.

In [19]:
import httpx
from acp_sdk.models import Message, MessagePart, Run


async def run_agent_sync(base_url: str, agent_name: str, query: str) -> Run:
    """Call an ACP agent synchronously using explicit JSON body semantics."""
    message = Message(parts=[MessagePart(content=query, content_type="text/plain")])
    payload = {
        "agent_name": agent_name,
        "input": [message.model_dump(mode="json")],
        "mode": "sync",
    }
    async with httpx.AsyncClient(base_url=base_url, timeout=60) as client:
        response = await client.post("/runs", json=payload)
        response.raise_for_status()
    return Run.model_validate(response.json())


async def run_query(user_query: str) -> str:
    """Send a query to the Advisor agent and return the response text."""
    print(f"Query: {user_query}")
    print("=" * 60)

    run = await run_agent_sync("http://localhost:8001", "advisor", user_query)

    texts = []
    for msg in run.output:
        for part in msg.parts:
            if hasattr(part, "content") and isinstance(part.content, str):
                texts.append(part.content)
    return "\n".join(texts)


# 8. Demo: Single Agent — Weather

The simplest case: the Advisor's LLM determines that only the `weather_agent` is needed.

In [25]:
import asyncio
# Running the query to the Advisor agent
response = asyncio.run(run_query("What's the weather in Tokyo?"))
print(response)

Query: What's the weather in Tokyo?
### Travel Advisory: Current Weather in Tokyo

**Current Conditions:**
- **Temperature:** 13.2°C
- **Humidity:** 46%
- **Wind Speed:** 3.1 km/h

As of now, the weather in Tokyo is mild, making it a comfortable time to explore the city. 

**7-Day Forecast:**
- **March 15:** High of 13.4°C, Low of 4.9°C, no rain expected.
- **March 16:** High of 12.8°C, Low of 6.3°C, no rain expected.
- **March 17:** High of 14.1°C, Low of 4.1°C, no rain expected.
- **March 18:** High of 14.7°C, Low of 5.2°C, no rain expected.
- **March 19:** High of 15.0°C, Low of 8.7°C, light rain (1.1 mm).
- **March 20:** High of 18.0°C, Low of 5.8°C, light rain (0.4 mm).
- **March 21:** High of 15.8°C, Low of 3.9°C, no rain expected.

### Recommendations:
- **Clothing:** Dress in layers to accommodate the varying temperatures throughout the day.
- **Activities:** Ideal weather for outdoor sightseeing, especially in parks and gardens. Consider visiting attractions like Ueno Park or 

# 9. Demo: Two-Agent Chain — Country + Weather

Now the Advisor must call **two** data agents — `country_agent` for languages and currency, and `weather_agent` for Paris conditions. Watch how it plans both calls and synthesizes the results.

In [26]:
import asyncio
# Query requiring Country and Weather agents
response = asyncio.run(run_query(
    "Tell me about France — what languages do they speak, what currency do they use, and what's the weather like in Paris?"
))
print(response)

Query: Tell me about France — what languages do they speak, what currency do they use, and what's the weather like in Paris?
### Travel Advisory: France

**Official Language:**  
The primary language spoken in France is **French**. While you may find some English speakers, especially in tourist areas, it's beneficial to learn a few basic French phrases to enhance your experience.

**Currency:**  
France uses the **euro (€)** as its official currency. Make sure to have some euros on hand for small purchases, although credit cards are widely accepted.

**Weather in Paris:**  
Currently, the weather in Paris is quite chilly, with a temperature of **2.2°C** and high humidity at **90%**. Over the next week, you can expect temperatures to range from **2.2°C to 19.5°C**. Here’s a brief overview of the upcoming weather:

- **March 15:** 2.2°C – 12.7°C, no rain
- **March 16:** 7.8°C – 13.7°C, no rain
- **March 17:** 5.1°C – 16.7°C, no rain
- **March 18:** 6.4°C – 19.5°C, no rain
- **March 19:**

# 10. Demo: Full Chain — Three Agents

This query requires all three data agents — `country_agent`, `weather_agent`, and `holiday_agent` — working together. The Advisor calls them **in parallel**, then synthesizes a complete travel brief.

In [27]:
import asyncio
# Full Chain Demo: Country + Weather + Holidays
response = asyncio.run(run_query(
    "I'm planning a trip to Japan. Give me a complete travel brief — "
    "country info, Tokyo weather, and upcoming public holidays for 2026."
))
print(response)

Query: I'm planning a trip to Japan. Give me a complete travel brief — country info, Tokyo weather, and upcoming public holidays for 2026.
### Travel Advisory: Trip to Japan

#### Country Overview
- **Official Name:** Japan
- **Capital:** Tokyo
- **Region:** Eastern Asia
- **Population:** Approximately 123.2 million
- **Language:** Japanese
- **Currency:** Japanese yen (¥)
- **Timezone:** UTC+09:00
- **Driving:** Left side of the road

#### Weather in Tokyo (March 2026)
During your visit in March 2026, you can expect mild temperatures in Tokyo. Here’s a brief forecast for the week of March 15-21, 2026:

- **March 15:** 4.9°C – 13.4°C, no rain
- **March 16:** 6.3°C – 12.8°C, no rain
- **March 17:** 4.1°C – 14.1°C, no rain
- **March 18:** 5.2°C – 14.7°C, no rain
- **March 19:** 8.7°C – 15.0°C, light rain (1.1 mm)
- **March 20:** 5.8°C – 18.0°C, light rain (0.4 mm)
- **March 21:** 3.9°C – 15.8°C, no rain

Overall, expect cool weather with a chance of light rain towards the end of the week

# 11. Demo: Open-Ended Reasoning

This is the most interesting case. The Advisor must decide **on its own** which agents to call and with what queries. To compare Brazil vs Thailand, it needs to call `country_agent` and `weather_agent` **twice each** — and the LLM must choose representative cities to check weather for.

In [28]:
import asyncio
# Open-ended reasoning demo
response = asyncio.run(run_query(
    "I'm deciding between Brazil and Thailand for a beach holiday. "
    "Compare both countries and their weather to help me decide."
))
print(response)

Query: I'm deciding between Brazil and Thailand for a beach holiday. Compare both countries and their weather to help me decide.
### Travel Advisory: Beach Holiday in Brazil vs. Thailand

When choosing between Brazil and Thailand for a beach holiday, both countries offer stunning coastlines, vibrant cultures, and unique experiences. Here’s a comparison to help you decide:

#### Country Overview

**Brazil**
- **Official Name:** Federative Republic of Brazil
- **Capital:** Brasília
- **Language:** Portuguese
- **Currency:** Brazilian real (R$)
- **Population:** Approximately 213 million
- **Timezone:** UTC-05:00 to UTC-02:00
- **Driving:** Right side

**Thailand**
- **Official Name:** Kingdom of Thailand
- **Capital:** Bangkok
- **Language:** Thai
- **Currency:** Thai baht (฿)
- **Population:** Approximately 66 million
- **Timezone:** UTC+07:00
- **Driving:** Left side

#### Weather Comparison

**Brazil (Rio de Janeiro)**
- **Current Temperature:** 21.2°C
- **Humidity:** 100%
- **7-Day F

# 12. Cleanup

In [30]:
# Terminate both agent servers
data_proc.terminate()
advisor_proc.terminate()
data_proc.wait()
advisor_proc.wait()
print("Both agent servers terminated.")

Both agent servers terminated.


# 13. Conclusion

In this notebook, we built a **multi-agent travel advisory system** using the Agent Communication Protocol:

- **Three Data Agents** (`weather_agent`, `country_agent`, `holiday_agent`) that autonomously wrap free public APIs
- **One Advisor Agent** that uses an LLM to plan, delegate, and synthesize
- **A thin client** in this notebook that sends one message and gets back a complete advisory

### Key Takeaways

1. **MCP = vertical, ACP = horizontal.** MCP connects an LLM to tools. ACP connects agents to each other. They're complementary, not competing.

2. **Agents are autonomous.** Each data agent independently decides how to fulfill requests — the Advisor sends natural language, not function parameters.

3. **The client is trivially simple.** Just one message in, one response out. All intelligence lives in the agent network.

4. **Agent discovery is the ACP equivalent of MCP tool listing.** The Advisor doesn't hardcode which agents exist — it discovers them at runtime via `GET /agents`.

5. **The LLM is an orchestrator, not a tool user.** Instead of calling functions, the LLM reasons about which *agents* to consult and what to ask them.

6. **Composition scales naturally.** Adding a new capability means deploying a new agent — no changes needed to the Advisor or client.

### Next Steps

- Add more data agents (flight prices, exchange rates, points of interest) — the Advisor discovers them automatically
- Deploy agents on separate machines to demonstrate true distributed agent networks
- Add streaming responses using ACP's `run_stream` for real-time output
- Build multi-hop chains where data agents can call other agents themselves
- Combine MCP and ACP: have data agents use MCP tools internally while exposing themselves via ACP